## Import liblary

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## Data Loading

In [2]:
df = pd.read_csv('/content/cleaned_data.csv')

In [3]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,previous_contact,pdays_clean
0,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,unknown,1,No Previous Contact,NaN
1,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,unknown,1,No Previous Contact,NaN
2,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,unknown,1,No Previous Contact,NaN
3,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,unknown,1,No Previous Contact,NaN
4,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,unknown,1,No Previous Contact,NaN


## Hypotesis Testing

#### 1. RQ1 — Does customer balance affect deposit subscription?
H0 : No significant difference
in balance between subscribed
and non-subscribed customers.

H1: There is a significant difference
in balance between groups.

In [4]:
deposit_yes = df[
    df['deposit'] == 1
]['balance']

deposit_no = df[
    df['deposit'] == 0
]['balance']

In [5]:
t_stat, p_value = stats.ttest_ind(
    deposit_yes,
    deposit_no,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 8.520420767660433
P-value: 1.8105789895875916e-17


Customers who subscribed to deposits exhibit significantly different account balances compared to non-subscribers (p < 0.05), indicating that financial capacity may influence deposit decisions.

#### 2. RQ2 — Does previous campaign outcome influence conversion?

H0 : There is no effect of the outcome of the previous campaign on deposits

H1: Yes, the results of the previous campaign do have an impact on deposits

In [6]:
cont_table = pd.crosstab(
    df['poutcome'],
    df['deposit']
)

cont_table

deposit,0,1
poutcome,,
failure,610,618
other,230,307
success,93,978
unknown,4940,3386


In [7]:
chi2, p, dof, expected = stats.chi2_contingency(
    cont_table
)

print("Chi-Square:", chi2)
print("P-value:", p)

Chi-Square: 1004.635780185333
P-value: 1.7761850102620281e-217


In [8]:
round(
    pd.crosstab(
        df['poutcome'],
        df['deposit'],
        normalize='index'
    ) * 100,
    2
)

deposit,0,1
poutcome,,
failure,49.67,50.33
other,42.83,57.17
success,8.68,91.32
unknown,59.33,40.67


The results of the previous campaign influenced customers to resubscribe, as indicated by a p-value of less than 0.05. As a result, the bank can focus on following up with customers who were successful in the previous campaign

#### 3. RQ3 — Does contact channel affect marketing success?

H0 : There is no significant correlation between contact channels and subscriptions

H1: There is a significant correlation between the two

In [9]:
cont_table = pd.crosstab(
    df['contact'],
    df['deposit']
)

cont_table

deposit,0,1
contact,,
cellular,3673,4369
telephone,384,390
unknown,1816,530


In [10]:
chi2, p, dof, expected = stats.chi2_contingency(
    cont_table
)

print("Chi-Square:", chi2)
print("P-value:", p)

Chi-Square: 736.6866796046972
P-value: 1.0728032438445805e-160


There is a significant relationship between contact channels and subscriptions, as indicated by a p-value of less than 0.05. Therefore, banks can make greater use of mobile contact channels, as previous results show that mobile channels generate higher deposit amounts.

#### RQ4 Which customer profile has the highest conversion probability?

In [14]:
features = [
    'balance',
    'campaign',
    'housing',
    'loan',
    'contact',
    'education',
    'marital',
    'poutcome',
    'job'
]

In [18]:
categorical_cols = [
    'job',
    'marital',
    'education',
    'contact',
    'month',
    'poutcome',
    'previous_contact'
]

In [19]:
df_model = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

In [26]:
X = df_model.drop([
    'deposit',
    'pdays',
    'campaign_group',
    'pdays_clean'

], axis=1)

y = df_model['deposit']

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LogisticRegression(
    max_iter=1000
)

model.fit(
    X_train,
    y_train
)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [28]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
})

coef_df.sort_values(
    by='Coefficient',
    ascending=False
)

,Feature,Coefficient
39,poutcome_success,2.042817
33,month_mar,1.261277
36,month_oct,0.998966
37,month_sep,0.797474
13,job_retired,0.491129
16,job_student,0.424064
23,education_tertiary,0.362228
28,month_dec,0.353029
32,month_jun,0.326809
22,education_secondary,0.204168


Customers with the highest conversion probability tend to be previously successful campaign responders, contacted through proper communication channels, possess higher education levels, and belong to student or retired customer groups. In contrast, customers with active housing or personal loans and those contacted excessively tend to show lower conversion likelihood.
